In [1]:
import gradio as gr
from transformers import pipeline

In [2]:
# 1. Instantiate an internal inference pipeline engine
# We use a highly optimized, compact model for fast interactive web inference
model_pipeline = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [3]:
# 2. Define the core prediction wrapper
# Gradio expects a standard Python function. It handles extracting inputs from the UI 
# and feeding outputs back to the page automatically.
def predict_sentiment(user_text):
    # Pass raw text input directly into the pipeline
    outputs = model_pipeline(user_text)
    
    # Extract structural items out of the pipeline wrapper response format:
    # [{'label': 'POSITIVE', 'score': 0.999}]
    first_prediction = outputs[0]
    predicted_label = first_prediction["label"]
    confidence_score = first_prediction["score"]
    
    # We return a dictionary format because Gradio's Label component 
    # maps dictionaries directly to visual confidence bar graphs!
    # e.g., {"POSITIVE": 0.99, "NEGATIVE": 0.01}
    other_label = "NEGATIVE" if predicted_label == "POSITIVE" else "POSITIVE"
    return {
        predicted_label: float(confidence_score),
        other_label: float(1.0 - confidence_score)
    }

In [4]:
# 3. Assemble and Configure the Gradio User Interface
demo_interface = gr.Interface(
    fn=predict_sentiment,              # The Python processing function defined above
    inputs=gr.Textbox(                 # UI Input Element: A rich text input area
        lines=3, 
        placeholder="Type your sentence here to test the model's sentiment analysis classification...",
        label="Input Text Fragment"
    ),
    outputs=gr.Label(num_top_classes=2, label="Model Confidence Output"), # UI Output Element: A specialized classification meter
    title="Production-Ready AI Sentiment Classification Hub",
    description="This web application interfaces directly with an underlying DistilBERT model. Type sentences to watch the classifications adjust in real-time.",
    examples=[
        ["This ML architecture roadmap is structured perfectly!"],
        ["I am struggling to get this codebase to execute without throwing errors."]
    ]
)

In [6]:
print("Gradio App prepared successfully.")
# 4. Launch the application server locally
# Setting share=True inside a local development terminal creates a temporary 
# public 'gradio.live' URL allowing anybody to visit your local server remotely!
demo_interface.launch()

Gradio App prepared successfully.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
